In [ ]:
__call__ 是 Python 的一种特殊方法，也叫“魔术方法”。它的作用是：让对象可以像函数一样，加括号调用。
例如：
class Adder:
    def __call__(self, a, b):
        return a + b

obj = Adder()     # 创建对象
result = obj(2, 3)  # Python 自动调用 __call__
print(result)      # 5
当你写：
obj(2, 3)
可以理解为 Python 自动执行了：
obj.__call__(2, 3)
这里的 self 指的是 obj，由 Python 自动传入；a 和 b 分别接收 2 和 3。
注意，方法名必须是 __call__，前后各两个下划线。这是 Python 约定好的名字，写成普通的 call 就没有这个自动触发的效果。
把它和你刚看到的 __init__ 放在一起：
特殊方法	什么时候自动执行	用途
__init__	创建对象、进行初始化时	设置对象的初始状态
__call__	对象后面加括号调用时	执行你定义的操作


对应你的代码：
criterion = nn.CrossEntropyLoss()  # 创建对象，初始化时执行 __init__
loss = criterion(logits, y)        # 调用对象，触发继承来的 __call__ 机制

用一个简化例子就能看清楚：
class Parent:
    def __call__(self, x):
        return self.forward(x)

class Child(Parent):
    def forward(self, x):
        return x * 2

obj = Child()
print(obj(3))  # 6
Child 没写 __call__，但继承了父类的 __call__，所以 obj(3) 仍然能执行。 父类里的 self 就是这个 obj，因此 self.forward(x) 会调用 Child 定义的 forward。
另外，注意区分这两处：
criterion = nn.CrossEntropyLoss()  # 创建对象，执行初始化 __init__
loss = criterion(logits, y)        # 调用对象，走 __call__ → forward
这里的 __call__ 是解释第二行为什么能这样写的。

# nn.CrossEntropyLoss() 这里为什么要加括号

因为 nn.CrossEntropyLoss 是一个类，加上 () 才会创建这个类的对象。
criterion = nn.CrossEntropyLoss    # 不加括号：criterion 指向这个类
criterion = nn.CrossEntropyLoss()  # 加上括号：创建对象，赋值给 criterion
可以把“类”理解成设计图，“对象”是按照设计图造出来的东西。
所以你的代码分成两步：
# 第一步：创建损失计算器
criterion = nn.CrossEntropyLoss()

# 第二步：用这个计算器计算损失
loss = criterion(logits, y)

# 为什么是 criterion(logits, y)？
计算误差需要两样东西：
logits = model(x)             # 模型的预测：每个类别的原始得分
loss = criterion(logits, y)   # y 是正确答案
例如，要把图片分成“猫、狗、鸟”三类，一个样本的输出可能是：
logits = torch.tensor([[2.0, 1.0, 0.1]])  # 猫、狗、鸟的原始得分
y = torch.tensor([0])                     # 正确类别是猫，编号为 0
对于这种用类别编号表示的标签，交叉熵的计算等价于：
先把得分转成概率 → 取出正确类别的概率 → 计算它的负对数。
原始得分：       [2.0,   1.0,   0.1]
Softmax 概率：  [0.659, 0.242, 0.099]
正确类别：      0，也就是猫
损失：          -ln(0.659) ≈ 0.417
公式就是：
\[
L = -\log p_{\text{正确类别}}
\]因此：
模型给正确类别的概率	损失
0.9	约 0.105
0.5	约 0.693
0.1	约 2.303


使用 CrossEntropyLoss 时直接传原始 logits，不用自己先做 Softmax。 它内部以数值更稳定的方式完成等价计算。

In [4]:
import torch.nn as nn
import torch
criterion = nn.CrossEntropyLoss()

class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(28*28, 256)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = mlp()
print(model.layer1.weight[:2, :5])  # 前两行、前五列的权重
print(model.layer1.bias[:5])       # 前五个偏置
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(64, 28*28)

for name, param in model.named_parameters():
    print(name, param.shape)

for epoch in range(3):
    optimizer.zero_grad()
    logits = model(x)
    loss = criterion(logits, torch.randint(0, 10, (64,)))
    loss.backward()
    optimizer.step()

    print(f"第 {epoch + 1} 轮，loss = {loss.item():.4f}")



tensor([[-0.0323,  0.0280,  0.0180,  0.0179,  0.0345],
        [-0.0160, -0.0130, -0.0234, -0.0177, -0.0311]],
       grad_fn=<SliceBackward0>)
tensor([-0.0164, -0.0289,  0.0161,  0.0209, -0.0257], grad_fn=<SliceBackward0>)
layer1.weight torch.Size([256, 784])
layer1.bias torch.Size([256])
layer2.weight torch.Size([10, 256])
layer2.bias torch.Size([10])
第 1 轮，loss = 2.3562
第 2 轮，loss = 2.3061
第 3 轮，loss = 2.2986


# self.layer2 = nn.Linear(256, 10)   最后一层的输出不是10嘛

对，最后一层对每个样本输出 10 个得分，但每个样本的正确答案只需要 1 个类别编号。
比如输入一张图片，模型输出：
# 类别编号： 0    1    2    3    4    5    6    7    8    9
logits = [0.2, 0.1, 1.3, 5.2, 0.7, 0.0, 0.3, 0.4, 0.8, 0.1]
这 10 个数表示模型给各个类别的得分。如果这张图片的正确类别是 3，标签就是：
y = 3
不需要给 10 个正确答案，只需要告诉损失函数：这 10 个类别中，第 3 类是正确的。
所以一次输入 64 个样本：
logits.shape  # (64, 10)：每个样本有 10 个预测得分
y.shape       # (64,)：每个样本有 1 个正确类别编号
对应你的代码：
self.layer2 = nn.Linear(256, 10)  # 每个样本输出 10 个类别得分
y = torch.randint(0, 10, (64,))   # 为 64 个样本各生成一个类别编号
loss = criterion(logits, y)      # 比较预测得分与正确类别
logits 和 y 含义不同，它们的形状不需要完全相同。


# 模型不知道答案，所以要给 10 个类别都打分；标签已经知道答案，所以只需要记录正确类别的编号。

1. 为什么最后一层必须输出 10 个数？

CIFAR-10 有 10 个类别。

假设：

self.layer2 = nn.Linear(256, 10)

这个 10 表示：

对于每一张图片，模型给 10 个类别分别打一个分数。

例如输入一张猫的图片：

          一张图片
             ↓
            MLP
             ↓
     最后一层 Linear
             ↓

类别：   0    1    2    3    4    5    6    7    8    9

得分：  0.2  0.1  1.3  5.2  0.7  0.0  0.3  0.4  0.8  0.1
                         ↑
                       最大

这 10 个数就是：

logits

如果类别 3 是 cat，那么模型现在明显更倾向于：

类别 3

因为：

$$ 5.2 $$

最大。

所以：

logits.argmax()

得到：

3
2. 那为什么 y 不也是 10 个数？

因为 y 不是模型预测。

y 是：

老师提前告诉我们的标准答案。

假设这张图片真实就是：

cat

而 CIFAR-10 规定：

cat → 类别编号 3

那么标准答案只需要：

y = 3

就够了。

因为：

“正确答案是第 3 类”

这句话已经把答案完整表达出来了。

没必要再写 10 个数字。

3. 用考试来类比最好理解

假设一道选择题：

A
B
C
D

模型不知道答案，所以它可能给每个选项一个分数：

模型：

A → 0.3
B → 1.2
C → 4.8
D → 0.5

模型实际上是在说：

“我觉得 C 最可能。”

所以模型需要输出：

[0.3, 1.2, 4.8, 0.5]

但是标准答案呢？

老师只需要告诉你：

正确答案：C

不需要写：

A 错
B 错
C 对
D 错

所以：

模型输出：4 个分数

标准答案：1 个编号

CIFAR-10 只是从 4 选 1 变成：

10 选 1
4. 所以 logits 和 y 本来就不是一种东西

这个区别特别重要：

logits
↓
模型的预测

y
↓
真实答案

对于一张图片：

模型预测 logits：

[0.2, 0.1, 1.3, 5.2, 0.7, 0.0, 0.3, 0.4, 0.8, 0.1]
 ↑                                                    ↑
 └──────────────── 10 个类别分数 ────────────────────┘


真实标签 y：

3

所以：

logits.shape = [10]

y = 一个数字
5. CrossEntropyLoss 怎么把它们比较起来？

这就是关键了。

假设：

logits = [0.2, 0.1, 1.3, 5.2, 0.7, ...]
y = 3

CrossEntropyLoss 看到：

y = 3

就知道：

哦，第 3 类才是真正正确的类别。

然后它重点关心：

模型有没有把第 3 类的相对分数弄得足够高？

概念上：

logits
  ↓
[0.2, 0.1, 1.3, 5.2, 0.7, ...]
                   ↑
                   │
y = 3 ─────────────┘
           正确类别

如果第 3 类的预测相对非常强：

类别 3 → 很高

Loss 就比较小。

如果第 3 类反而很低：

类别 3 → 很低
其他错误类别 → 很高

Loss 就很大。

6. 如果一次输入 64 张图片呢？

现在 Batch Size：

batch_size = 64

模型一次处理 64 张图片。

每张图片输出：

10 个 logits

所以：

logits.shape

就是：

$$ \boxed{[64,10]} $$

可以想成一张表：

                 10 个类别
              0    1    2    3   ...   9

图片0       [ 0.2  0.1  1.3  5.2  ... 0.1 ]
图片1       [ 2.1  0.3  0.1  0.5  ... 1.2 ]
图片2       [ 0.1  4.2  0.3  0.2  ... 0.7 ]
图片3       [ 0.4  0.2  3.8  0.1  ... 0.2 ]
...
图片63      [ 1.2  0.1  0.4  0.8  ... 2.7 ]

 ↑
64 张图片

所以：

[64, 10]
 ↑    ↑
 │    └── 每张图片 10 个类别得分
 │
 └─────── 64 张图片
7. 那 64 张图片需要多少个真实答案？

每张图片只需要 一个正确类别编号。

假设：

图片0 → 正确类别 3
图片1 → 正确类别 0
图片2 → 正确类别 1
图片3 → 正确类别 2
...
图片63 → 正确类别 9

那么：

y

就是：

[3, 0, 1, 2, ..., 9]

一共有：

64 个数字

所以：

y.shape

是：

$$ \boxed{[64]} $$
8. 把两个 shape 放一起看

现在应该是最关键的一步：

logits.shape = [64, 10]
                 ↑    ↑
                 │    └── 每张图片 10 个预测分数
                 │
                 └─────── 64 张图片


y.shape      = [64]
                ↑
                └──────── 每张图片 1 个正确类别

所以：

             模型预测                         正确答案

图片0    [10 个类别得分]     ←比较→          3
图片1    [10 个类别得分]     ←比较→          0
图片2    [10 个类别得分]     ←比较→          1
图片3    [10 个类别得分]     ←比较→          2
...
图片63   [10 个类别得分]     ←比较→          9

这就是：

loss = criterion(logits, y)

在干的事情。

9. torch.randint(0, 10, (64,)) 又是什么意思？

你代码里：

y = torch.randint(0, 10, (64,))

我们拆开：

0
↓
最小类别编号

10
↓
上界，不包含 10

(64,)
↓
生成 64 个数

所以会随机得到类似：

tensor([
    3, 7, 0, 1, 9, 2, 4, 3, ...
])

每个数字范围：

$$ 0\sim9 $$

刚好对应 CIFAR-10 的 10 个类别。

所以：

y.shape
# torch.Size([64])
10. 你可能真正困惑的是：为什么不使用 one-hot？

你可能直觉上觉得：

既然模型输出：

[0.2, 0.1, 1.3, 5.2, ...]

那么标准答案是不是也应该写成：

[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]

这个想法其实完全合理。

这叫：

$$ \boxed{\text{One-Hot Encoding}} $$

如果真实类别是 3：

类别：
0 1 2 3 4 5 6 7 8 9

标签：
0 0 0 1 0 0 0 0 0 0
      ↑
   第3类正确

所以：

y = 3

和：

y_one_hot = [0,0,0,1,0,0,0,0,0,0]

在表达“哪个类别正确”这件事情上，包含的是相同的类别信息。

但对于普通单标签分类，PyTorch 的：

nn.CrossEntropyLoss()

常见用法允许你直接给：

y = 3

所以没必要自己转成 one-hot。

这样反而更简单：

真实类别是 3

直接：
3

而不是：
[0,0,0,1,0,0,0,0,0,0]
11. 最后一层的 10 和标签的 1 不矛盾

这是你现在最需要打通的地方。

self.layer2 = nn.Linear(256, 10)

这里的：

10

回答的是：

模型有多少个类别需要选择？

而：

y = 3

回答的是：

这张图片真正属于哪一个类别？

它们回答的是两个完全不同的问题。

所以：

模型：
“10 个类别我都不知道哪个对，
 所以我给每一个类别打分。”

       ↓

[0.2, 0.1, 1.3, 5.2, 0.7, ...]
                    ↑
                 模型觉得3最好


老师：
“我知道答案，我只告诉你正确类别编号。”

       ↓

       y = 3

然后：

                CrossEntropyLoss
                       ↓

模型预测： [0.2 0.1 1.3 5.2 ...]
                          ↑
                          │
真实答案：                3
                          │
                          ↓
               看模型有没有把
               正确类别预测得好
最后记这一句话

你以后看到：

logits.shape = [64, 10]
labels.shape = [64]

不要觉得奇怪。

直接翻译：

$$ \boxed{ [64,10] = 64个样本\times每个样本10个类别预测分数 } $$

而：

$$ \boxed{ [64] = 64个样本\times每个样本1个正确类别编号 } $$

最核心就是：

预测需要对所有可能的类别打分，所以有 10 个；答案只需要告诉你哪个类别是正确的，所以只有 1 个编号。

这个理解透以后，你后面学 CrossEntropyLoss、argmax、accuracy 都会一下顺很多。


# model = MLP()

这一步的核心含义是：

根据你定义的 MLP 类，创建一个真正的神经网络对象 model。

假设你的类是：

import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(2, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

那么执行：

model = MLP()

主要发生了下面几件事。

首先，Python 会创建一个 MLP 类的实例，也就是一个对象。可以理解成：

MLP = 模型设计图纸
model = 按照图纸造出来的实际模型

就像：

class Dog:
    ...

dog = Dog()

Dog 是类，dog 是具体对象。

对于神经网络也是一样：

class MLP(nn.Module):
    ...

是“模型模板”，而：

model = MLP()

才是真正可以拿来训练的那个模型。

然后，创建对象的时候，会自动执行：

MLP.__init__()

也就是：

def __init__(self):
    super().__init__()

    self.fc1 = nn.Linear(2, 16)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(16, 2)

所以实际上你可以把：

model = MLP()

理解成触发了：

创建 MLP 对象
    ↓
执行 __init__()
    ↓
创建 fc1
    ↓
创建 ReLU
    ↓
创建 fc2
    ↓
把它们注册到 model 中

这里特别重要的是 nn.Linear 并不只是“登记一下层”。

例如：

self.fc1 = nn.Linear(2, 16)

执行的时候，PyTorch 会真正创建这一层的参数：

fc1.weight
fc1.bias

其中：

model.fc1.weight.shape

是：

[16, 2]

而：

model.fc1.bias.shape

是：

[16]

这些参数通常还会被随机初始化。

所以执行完：

model = MLP()

以后，这个模型已经拥有了一套真实的参数。

例如你可以：

print(model.fc1.weight)

看到一堆数：

tensor([
    [ 0.231, -0.442],
    [ 0.518,  0.173],
    ...
])

这些就是以后训练时会被更新的参数。

同理：

self.fc2 = nn.Linear(16, 2)

会创建：

fc2.weight: [2, 16]
fc2.bias:   [2]

所以你的整个模型创建完成后，大概是：

model
│
├── fc1
│    ├── weight [16, 2]
│    └── bias   [16]
│
├── relu
│    └── 没有可训练参数
│
└── fc2
     ├── weight [2, 16]
     └── bias   [2]

这里还有一个特别容易混淆的点：

model = MLP()

不会执行 forward()。

也就是说：

model = MLP()

只是：

创建模型、创建网络层、初始化参数。

还没有任何数据进入网络。

真正执行 forward() 是后面的：

output = model(x)

这时候才会发生：

x
↓
fc1
↓
ReLU
↓
fc2
↓
output

所以你可以这样区分：

model = MLP()

相当于：

造机器。

而：

output = model(x)

相当于：

把原料 x 放进机器里运行。

再具体一点：

class MLP(nn.Module):
    ...

是在：

设计机器。

model = MLP()

是在：

制造机器。

output = model(x)

是在：

使用机器。

还有一点非常关键：因为你的 MLP 继承了：

nn.Module

所以当执行：

model = MLP()

之后，PyTorch 会自动管理这些子模块和参数。

例如：

for name, param in model.named_parameters():
    print(name, param.shape)

会自动找到：

fc1.weight  [16, 2]
fc1.bias    [16]
fc2.weight  [2, 16]
fc2.bias    [2]

所以后面你才能直接：

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

因为 model.parameters() 已经知道这个模型里有哪些可训练参数。

你现在最需要记住这个完整流程：

class MLP(nn.Module)
        ↓
定义模型模板

model = MLP()
        ↓
执行 __init__()
        ↓
创建 Linear / ReLU
        ↓
初始化 weight / bias
        ↓
注册 Parameter
        ↓
得到一个真正的模型对象

model(x)
        ↓
才执行 forward(x)
        ↓
得到模型输出

一句话总结：

model = MLP() 是“实例化模型”：创建 MLP 对象、执行 __init__()、创建并初始化各层及其参数，但此时还没有进行前向传播；只有执行 model(x) 才会调用 forward()。

# model = mlp()每次调用这个函数，他的初始的权重是一样的嘛
不一样。每次调用 mlp() 都会重新创建一层新的网络，权重是独立随机初始化的，两次调用的结果通常不同。

1. 为什么会不一样
PyTorch 里 nn.Linear、nn.Conv2d 等层在创建时就会初始化权重，用的是随机初始化（比如 Kaiming、Xavier）。

python
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 10)

model1 = MLP()
model2 = MLP()

print(torch.equal(model1.fc.weight, model2.fc.weight))  # False
每次 MLP() 都会执行 __init__，里面 nn.Linear(...) 重新随机初始化，所以两个模型权重不同。

2. 但“随机”不是完全不可控
PyTorch 的随机数来自全局随机种子（seed）。

不设种子
python
model1 = MLP()
model2 = MLP()
# 两次权重不同
因为全局 RNG 状态在变。

设了种子
python
torch.manual_seed(42)
model1 = MLP()

torch.manual_seed(42)
model2 = MLP()

print(torch.equal(model1.fc.weight, model2.fc.weight))  # True
只要在创建前把种子重置成同一个值，两次初始化就完全一样。

关键点：

种子控制的是随机数序列

重置种子 → 序列从头开始 → 同样的随机数 → 同样的初始化

不重置 → 序列继续往前走 → 不同的随机数

3. 连续创建、不重置种子
python
torch.manual_seed(42)

model1 = MLP()   # 用序列前一段随机数
model2 = MLP()   # 用序列后一段随机数
即使设了种子，连续创建两个模型，权重也不同，因为第二次用的是序列里后面的随机数。

4. 什么时候需要“权重一样”
场景	做法
复现实验	训练开始前设种子
对比两个模型	用同一个种子分别初始化
加载已训练模型	用 load_state_dict，不靠随机初始化
集成学习	故意用不同种子，让模型多样化
复现实验的标准写法
python
import torch
import numpy as np
import random

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)
model = MLP()
这样每次运行脚本，初始化都一样。

5. 加载权重时不受初始化影响
python
model = MLP()                          # 随机初始化
model.load_state_dict(torch.load("best.pth"))  # 覆盖成训练好的权重
load_state_dict 会把随机初始化的值全部覆盖，所以初始化是什么无所谓。

6. 常见误解
误解	事实
每次调用 mlp() 权重一样	默认不一样
设了种子后连续调用也一样	不一样，除非每次重置种子
权重是零初始化	不是，通常随机小值
初始化不重要	很重要，影响训练收敛
7. 一句话总结
每次调用 mlp() 都会重新随机初始化权重，默认两次结果不同；只有在创建前重置同一个随机种子，两次初始化才会完全一致。加载 state_dict 时初始化值会被覆盖，所以不影响。

